In [4]:
"""
Step 1: 硬件连接与手动控制
===========================
功能：
- 连接远程硬件服务 (SLM + 电动台 + 旋转台)
- 检查当前位置状态
- 上传测试相位图验证 SLM 连接
- 设置旋转台到正确角度

注意：运行前请确保 RPyC 硬件服务已启动 (端口 18861)
"""

from hardware import RemoteHardwareManager
import numpy as np

# ============================================================
# 连接远程硬件服务
# ============================================================
# RemoteHardwareManager 统一管理所有硬件：
#   - SLM 控制
#   - Z 轴平移台 (Z825B)
#   - 旋转台 (PRM1-Z8)
#   - AutoHotkey 鼠标控制（用于触发相机）
hw = RemoteHardwareManager(host="127.0.0.1", port=18861)

# ============================================================
# 检查当前位置
# ============================================================
# 获取 Z 轴位置 (单位: mm)
current_pos = hw.stage_get_position()
print(f"📍 当前 Z 轴位置: {current_pos:.4f} mm")

# 获取旋转台角度 (单位: 度)
angle = hw.rotation_get_position()
print(f"🔄 当前旋转台角度: {angle:.2f} °")

# ============================================================
# 测试 SLM 连接：上传 Fresnel 测试图案
# ============================================================
from phase_generators import PhaseGenerator
from optics_utils import load_dict_from_json

# 加载配置并生成测试相位图
params = load_dict_from_json(r".\\config\\base.json")
params['M'] = 5  # 设置为 5x5 微透镜阵列

Optimizer = PhaseGenerator(params)
Optimizer.generate(mode='fresnel')  # 生成 Fresnel 透镜相位
phase_8bit = Optimizer.update_phase_8bit()

# 上传到 SLM
hw.upload_slm(phase_8bit)
print(f"✅ 测试图案 (M={params.get('M', 'default')}) 已上传到 SLM")

# ============================================================
# 常用命令参考（取消注释以使用）
# ============================================================

# --- Z 轴平移台控制 ---
# 回零点:
# hw.stage_home()

# 移动到指定位置 (mm):
# hw.stage_move_to(11.805)
# pos = hw.stage_get_position()
# print(f"位置: {pos:.4f} mm")

# ============================================================
# 旋转台角度预设
# ============================================================
# 不同 M 值对应的标准角度（根据实际光路校准）
M_ANGLE = {
    'M3': 282.75,  # 3x3 阵列对应角度
    'M5': 268.0,   # 5x5 阵列对应角度
    'M7': 261.6,   # 7x7 阵列对应角度
    'M9': 258.5    # 9x9 阵列对应角度
}

# 设置旋转台到目标角度
target_angle = M_ANGLE['M5']
# 先超调再回位，避免齿轮间隙误差
hw.rotation_move_to(target_angle + 10)
hw.rotation_move_to(target_angle)
print(f"🔄 旋转台已移动到: {hw.rotation_get_position():.2f} °")

Connecting to local hardware service at 127.0.0.1:18861 ...
✅ Connected to unified hardware service
   Stage status:
     - Stage 1 (PRM1-Z8 (Rotation)): ✅ Connected
     - Stage 2 (Z825B (Z-Axis)): ✅ Connected
📍 当前 Z 轴位置: 12.0050 mm
🔄 当前旋转台角度: 268.00 °
Using device: cuda
F/47.31, 73.89999999999999 mm
Lens width: 1.562mm
Airy radius: 29.7um
Depth of focus: 2.31mm
Multi-depth planes are used at F=72.7 (-0.5DOF), 75.1 (0.5DOF) mm
Fresnel Lens: 850x850 px; 5x5 lenses
🚀 Phase pattern sent to SLM
✅ 测试图案 (M=5) 已上传到 SLM
🔄 旋转台已移动到: 268.00 °


## 测试不同配置的Fresnel

In [ ]:
from phase_generators import PhaseGenerator
from optics_utils import load_dict_from_json
from hardware import RemoteHardwareManager

# 加载配置并生成测试相位图
params = load_dict_from_json(r".\\config\\base.json")
params['M'] = 9  # 设置微透镜阵列规模：5x5 / 7x7 / 9x9

# 生成 Fresnel 透镜相位；上传到 SLM
Optimizer = PhaseGenerator(params)
Optimizer.generate(mode='fresnel') 
phase_8bit = Optimizer.update_phase_8bit()
hw.upload_slm(phase_8bit)
print(f"✅ 测试图案 (M={params.get('M', 'default')}) 已上传到 SLM")

# 设置旋转台到目标角度
label = 'M' + str(params['M']) # 例如 'M5'
target_angle = M_ANGLE[label]
# 先超调再回位，避免齿轮间隙误差
hw.rotation_move_to(target_angle + 10)
hw.rotation_move_to(target_angle)
print(f"🔄 旋转台已移动到: {hw.rotation_get_position():.2f} °")

Using device: cuda
F/85.15, 73.89999999999999 mm
Lens width: 0.868mm
Airy radius: 53.5um
Depth of focus: 7.47mm
Multi-depth planes are used at F=70.2 (-0.5DOF), 77.6 (0.5DOF) mm
Fresnel Lens: 850x850 px; 9x9 lenses
🚀 Phase pattern sent to SLM
✅ 测试图案 (M=9) 已上传到 SLM
🔄 旋转台已移动到: 258.50 °


# PSF 自动采集工作流程

## 概述
本 Notebook 用于自动化采集点扩散函数 (PSF) 数据，配合 SLM 相位图进行 Z 轴扫描成像。

## 工作流程
1. **硬件连接** - 连接 SLM、电动平移台、旋转台，测试通信
2. **选择相位图** - 从 `./output/` 选择要采集的 `.npy` 相位图文件
3. **Z-Scan 采集** - 自动执行 Z 轴扫描，触发相机采集
4. **整理数据** - 按相位图名称整理 TIFF 文件，生成元数据

## 硬件要求
- SLM (Meadowlark)
- Z 轴电动平移台 (Thorlabs Z825B)
- 旋转台 (Thorlabs PRM1-Z8)
- 相机软件 (通过 AutoHotkey 触发)
- RPyC 硬件服务 (端口 18861)

## 数据保存位置
默认保存到 NAS: `Z:\SLM_super_resolution\data\for_auto_scan\`

---

In [3]:
"""
Step 2: 选择相位图文件
=======================
功能：
- 扫描 ./output/ 目录下的 .npy 相位图文件
- 提供多选 GUI 界面
- 自动解析文件名中的 M 值 (如 M5, M7)

使用说明：
- Ctrl + 点击 可多选文件
- 文件名需包含 M%d 模式才会被识别（如 opt1_M5_xxx.npy）
"""

from npy_file_selector import select_npy_files

# ============================================================
# 创建文件选择器 GUI
# ============================================================
# select_npy_files 参数说明：
#   output_dir: 搜索 .npy 文件的目录

file_selector_widget = select_npy_files(output_dir="./output")

# ============================================================
# 选择完成后，可通过以下方法获取选择结果：
# ============================================================
# 获取选中的文件名列表：
#   selected_files = file_selector_widget.get_selected_files()
#
# 获取对应的 M 值列表（如 ['M5', 'M5', 'M7']）：
#   m_patterns = file_selector_widget.get_m_patterns()
#
# 获取输出目录路径：
#   output_dir = file_selector_widget.get_output_dir()

✅ Found 24 .npy files in output
Select .npy files for scanning (Ctrl+Click for multiple):
Files without M%d pattern will be automatically excluded.



In [ ]:
"""
Step 3: Z-Scan 自动采集
========================
功能：
- 自动识别相位图的 M 值（M3/M5/M7/M9）
- 根据 M 值自动选择对应的扫描参数（num_steps, z_range）
- 扫描前自动设置旋转台角度和上传 SLM 相位图
- 通过 AutoHotkey 触发相机采集
- 记录完整扫描参数供后续分析

注意事项：
- 运行前请确保相机软件已打开并处于待触发状态
- 第一次运行会提示手动点击相机触发按钮以捕获位置
- 扫描过程中请勿操作鼠标
"""

from hardware import RemoteHardwareManager
import numpy as np
import time
import os
from datetime import datetime
from nas_mapper import quick_map

# ============================================================
# NAS 网络驱动器映射（如果保存到 NAS）
# ============================================================
success, msg = quick_map()
if not success:
    raise RuntimeError(f"❌ NAS 映射失败: {msg}")

# ============================================================
# 扫描参数配置 ★★★ 可根据需要修改 ★★★
# ============================================================

# 数据保存目录（Z: 盘 = NAS 路径）
save_dir = r"Z:\\SLM_super_resolution\\data\\for_auto_scan\\"

# 焦平面位置 (mm) - Z 扫描的中心位置
z_focal_plane = 11.805

# ----- 不同 M 值对应的扫描参数 -----
# 格式: 'M{n}': {'num_steps': 步数, 'z_range': 扫描范围(mm)}
# 注意: 较大的 M 值需要更大的扫描范围以覆盖更大的景深
SCAN_PARAMS = {
    'M3': {'num_steps': 81, 'z_range': 0.3},   # 3x3 阵列
    'M5': {'num_steps': 81, 'z_range': 0.4},   # 5x5 阵列
    'M7': {'num_steps': 81, 'z_range': 0.6},   # 7x7 阵列
    'M9': {'num_steps': 81, 'z_range': 0.7},   # 9x9 阵列
}

# ----- 不同 M 值对应的旋转台角度 -----
# 根据实际光路校准确定
M_ANGLE = {
    'M3': 282.75,
    'M5': 268.0,
    'M7': 261.6,
    'M9': 258.5,
}

# ============================================================
# 从 Step 2 获取选中的文件信息
# ============================================================
# get_selected_files() 返回文件名列表（用于显示和记录）
# get_selected_paths() 返回完整路径列表（用于实际加载文件）
# get_m_patterns() 返回 M 值列表
selected_npy_files = file_selector_widget.get_selected_files()  # 文件名列表
selected_npy_paths = file_selector_widget.get_selected_paths()  # 完整路径列表 ★ 修复关键
m_list = file_selector_widget.get_m_patterns()                  # M 值列表 (如 ['M5', 'M5', 'M7'])

# ============================================================
# 验证文件选择
# ============================================================
if not selected_npy_files:
    raise ValueError("❌ 未选择任何 .npy 文件！请返回 Step 2 选择文件。")

# 验证所有 M 值都有对应的扫描参数
for m in set(m_list):
    if m not in SCAN_PARAMS:
        raise ValueError(f"❌ 未知的 M 值: {m}，请在 SCAN_PARAMS 中添加配置")
    if m not in M_ANGLE:
        raise ValueError(f"❌ 未知的 M 值: {m}，请在 M_ANGLE 中添加配置")

print(f"📁 已选择 {len(selected_npy_files)} 个相位图文件:")
print(f"{'─'*60}")
for i, (name, path, m) in enumerate(zip(selected_npy_files, selected_npy_paths, m_list)):
    params = SCAN_PARAMS[m]
    print(f"   [{i+1}] {name}")
    print(f"       路径: {path}")
    print(f"       → {m}: {params['num_steps']} 步, ±{params['z_range']/2:.2f} mm, 角度 {M_ANGLE[m]}°")

# ============================================================
# 计算总帧数（不同 M 可能有不同步数）
# ============================================================
total_frames = sum(SCAN_PARAMS[m]['num_steps'] for m in m_list)
print(f"\n📊 扫描统计:")
print(f"   总相位图数: {len(selected_npy_files)}")
print(f"   总帧数: {total_frames}")

# 按 M 值统计
m_counts = {}
for m in m_list:
    m_counts[m] = m_counts.get(m, 0) + 1
for m, count in sorted(m_counts.items()):
    params = SCAN_PARAMS[m]
    print(f"   {m}: {count} 个相位图 × {params['num_steps']} 步 = {count * params['num_steps']} 帧")

# ============================================================
# 连接硬件
# ============================================================
hw = RemoteHardwareManager(host="127.0.0.1", port=18861)

# ============================================================
# 捕获相机触发按钮位置
# ============================================================
print("\n🎯 正在捕获相机触发按钮位置...")
print("   请在相机软件上点击【拍照/采集】按钮...")
click_pos = hw.capture_position()
if click_pos is None:
    raise RuntimeError("❌ 捕获点击位置失败！")

# ============================================================
# 初始化扫描记录
# ============================================================
scan_start_time = datetime.now()
scan_info = {
    'start_time': scan_start_time.isoformat(),
    'z_focal_plane': z_focal_plane,
    'scan_params_by_m': SCAN_PARAMS,
    'm_angles': M_ANGLE,
    'patterns': [],  # 每个 pattern 的详细信息
    'save_dir': save_dir,
}

# ============================================================
# 清理临时文件夹中的旧 TIFF 文件
# ============================================================
old_files = [f for f in os.listdir(save_dir) if f.endswith((".tiff", ".tif"))]
for file in old_files:
    os.remove(os.path.join(save_dir, file))
print(f"🧹 已清理 {len(old_files)} 个旧 TIFF 文件")

# ============================================================
# 主扫描循环
# ============================================================
frame_counter = 0
current_m = None  # 跟踪当前 M 值，用于判断是否需要切换旋转台

print(f"\n{'='*60}")
print(f"🚀 开始扫描")
print(f"{'='*60}")

for pattern_idx, (npy_name, npy_path) in enumerate(zip(selected_npy_files, selected_npy_paths)):
    # --- 获取当前相位图的 M 值和对应参数 ---
    m_pattern = m_list[pattern_idx]
    params = SCAN_PARAMS[m_pattern]
    num_steps = params['num_steps']
    z_range = params['z_range']
    target_angle = M_ANGLE[m_pattern]
    
    # 计算该 pattern 的 Z 位置序列
    z_positions = np.linspace(
        z_focal_plane - z_range / 2,
        z_focal_plane + z_range / 2,
        num_steps
    )
    z_step = z_positions[1] - z_positions[0] if len(z_positions) > 1 else 0
    
    print(f"\n[相位图 {pattern_idx+1}/{len(selected_npy_files)}] {npy_name}")
    print(f"   {m_pattern}: {num_steps} 步, 范围 ±{z_range/2:.3f} mm, 步长 {z_step*1000:.2f} μm")
    
    # ============================================================
    # 扫描前设置：旋转台角度（仅当 M 值变化时切换）
    # ============================================================
    if m_pattern != current_m:
        print(f"   🔄 切换旋转台: {current_m} → {m_pattern} (目标角度: {target_angle}°)")
        hw.rotation_move_to(target_angle + 10)  # 先超调，消除齿轮间隙
        hw.rotation_move_to(target_angle)
        actual_angle = hw.rotation_get_position()
        print(f"   🔄 旋转台已就位: {actual_angle:.2f}°")
        current_m = m_pattern
    
    # ============================================================
    # 扫描前设置：上传相位图到 SLM
    # ============================================================
    # 使用完整路径加载文件 ★ 修复关键
    pattern = np.load(npy_path)
    hw.upload_slm(pattern)
    print(f"   ✅ 相位图已上传到 SLM (从 {npy_path})")
    
    # ============================================================
    # 记录该 pattern 的扫描信息
    # ============================================================
    pattern_info = {
        'name': npy_name,
        'path': npy_path,
        'm_pattern': m_pattern,
        'num_steps': num_steps,
        'z_range': z_range,
        'z_positions': z_positions.tolist(),
        'z_step': z_step,
        'rotation_angle': target_angle,
        'frame_start': frame_counter + 1,  # 该 pattern 的起始帧号
    }
    
    # --- 初始化 Z 轴位置 ---
    hw.stage_move_to(z_positions[0])
    time.sleep(0.2)

    # ============================================================
    # Z 轴扫描子循环
    # ============================================================
    for z_idx, z_pos in enumerate(z_positions):
        frame_counter += 1
        
        # 移动 Z 轴
        hw.stage_move_to(z_pos)
        time.sleep(0.2)  # 等待电机稳定
        
        # 触发相机采集
        hw.click_at()
        time.sleep(0.65)  # 等待采集完成
        
        # 显示进度
        progress = frame_counter / total_frames * 100
        print(f"\r   Z[{z_idx+1}/{num_steps}] = {z_pos:.4f} mm | "
              f"帧 {frame_counter}/{total_frames} ({progress:.1f}%)", end='')
    
    print()  # 换行
    
    # 记录该 pattern 的结束帧号
    pattern_info['frame_end'] = frame_counter
    scan_info['patterns'].append(pattern_info)

# ============================================================
# 扫描完成，记录结束时间
# ============================================================
scan_end_time = datetime.now()
scan_info['end_time'] = scan_end_time.isoformat()
scan_info['duration_seconds'] = (scan_end_time - scan_start_time).total_seconds()
scan_info['total_frames'] = frame_counter

print(f"\n{'='*60}")
print(f"✅ 扫描完成！")
print(f"   完成时间: {scan_end_time.strftime('%Y-%m-%d %H:%M:%S')}")
print(f"   总耗时: {scan_info['duration_seconds']:.1f} 秒")
print(f"   采集帧数: {frame_counter}")
print(f"\n📊 各 M 值扫描汇总:")
for m in sorted(set(m_list)):
    params = SCAN_PARAMS[m]
    count = m_list.count(m)
    print(f"   {m}: {count} 个相位图, 步数={params['num_steps']}, 范围=±{params['z_range']/2:.2f}mm")
print(f"{'='*60}")

In [ ]:
"""
Step 4: 整理采集数据
=====================
功能：
- 根据 scan_info 中记录的每个 pattern 信息整理 TIFF 文件
- 支持不同 M 值有不同的帧数
- 文件名包含 M 值、Z 位置等完整信息
- 生成 JSON 格式的扫描元数据文件

输出结构：
  save_dir/
  ├── {pattern_name}/
  │   ├── {prefix}_{M}_frame001_z11.6050mm.tiff
  │   ├── {prefix}_{M}_frame002_z11.6100mm.tiff
  │   ├── ...
  │   └── {pattern_name}_scan_info.json
  ├── {pattern_name_2}/
  │   └── ...
  └── scan_log_{timestamp}.json   # 总扫描日志
"""

import os
import glob
import shutil
import json
from datetime import datetime

# ============================================================
# 用户参数设置
# ============================================================
# 文件名前缀（描述样品或实验条件）
user_prefix = "PSF_4um"

# ============================================================
# 使用 Step 3 中的扫描信息
# ============================================================
# 需要从上一个 cell 继承：scan_info, save_dir

print(f"📂 整理数据目录: {save_dir}")
print(f"   文件名前缀: {user_prefix}")

# ============================================================
# 查找采集的 TIFF 文件
# ============================================================
tiff_pattern = os.path.join(save_dir, "ss_single_*.tiff")
tiff_files = sorted(
    glob.glob(tiff_pattern), 
    key=lambda x: int(os.path.basename(x).replace('ss_single_', '').replace('.tiff', ''))
)

expected_frames = scan_info['total_frames']
print(f"   找到 {len(tiff_files)} 个 TIFF 文件 (预期: {expected_frames})")

if len(tiff_files) != expected_frames:
    print(f"⚠️ 警告: 文件数量不匹配！")

# ============================================================
# 按相位图整理文件（使用 scan_info 中的详细信息）
# ============================================================
frame_idx = 0

for pattern_idx, pattern_info in enumerate(scan_info['patterns']):
    npy_name = pattern_info['name']
    m_pattern = pattern_info['m_pattern']
    num_steps = pattern_info['num_steps']
    z_positions = pattern_info['z_positions']
    z_range = pattern_info['z_range']
    z_step = pattern_info['z_step']
    
    # 创建以相位图名称命名的子文件夹
    pattern_basename = os.path.splitext(npy_name)[0]
    pattern_folder = os.path.join(save_dir, pattern_basename)
    os.makedirs(pattern_folder, exist_ok=True)
    
    print(f"\n[{pattern_idx+1}/{len(scan_info['patterns'])}] {pattern_basename}/")
    print(f"   {m_pattern}: {num_steps} 帧, 范围 ±{z_range/2:.3f} mm")
    
    # --- 移动并重命名 TIFF 文件 ---
    moved_count = 0
    for z_idx, z_pos in enumerate(z_positions):
        if frame_idx >= len(tiff_files):
            print(f"   ⚠️ 缺失帧 {frame_idx+1}")
            frame_idx += 1
            continue
        
        src_path = tiff_files[frame_idx]
        
        # 新文件名格式: {prefix}_{M}_frame{n}_z{position}mm.tiff
        # 包含 M 值以便于后续处理
        new_name = f"{user_prefix}_{m_pattern}_frame{z_idx+1:03d}_z{z_pos:.4f}mm.tiff"
        dst_path = os.path.join(pattern_folder, new_name)
        
        shutil.move(src_path, dst_path)
        frame_idx += 1
        moved_count += 1
    
    print(f"   ✅ 已移动 {moved_count} 帧")
    
    # --- 生成该相位图的扫描信息文件 ---
    info_filename = f"{pattern_basename}_scan_info.json"
    info_path = os.path.join(pattern_folder, info_filename)
    
    pattern_scan_info = {
        'pattern_name': npy_name,
        'm_pattern': m_pattern,
        'user_prefix': user_prefix,
        'scan_start_time': scan_info['start_time'],
        'scan_end_time': scan_info['end_time'],
        'duration_seconds': scan_info['duration_seconds'],
        'z_focal_plane_mm': scan_info['z_focal_plane'],
        'z_range_mm': z_range,
        'num_steps': num_steps,
        'z_positions_mm': z_positions,
        'z_step_mm': z_step,
        'rotation_angle_deg': pattern_info['rotation_angle'],
        'frame_range': [pattern_info['frame_start'], pattern_info['frame_end']],
        'file_naming': f"{user_prefix}_{m_pattern}_frame{{n:03d}}_z{{z:.4f}}mm.tiff",
        'total_frames': num_steps,
    }
    
    with open(info_path, 'w', encoding='utf-8') as f:
        json.dump(pattern_scan_info, f, indent=2, ensure_ascii=False)
    
    print(f"   ✅ 已保存 {info_filename}")

# ============================================================
# 生成总扫描日志
# ============================================================
master_log_path = os.path.join(save_dir, f"scan_log_{scan_start_time.strftime('%Y%m%d_%H%M%S')}.json")

master_info = {
    'scan_start_time': scan_info['start_time'],
    'scan_end_time': scan_info['end_time'],
    'duration_seconds': scan_info['duration_seconds'],
    'user_prefix': user_prefix,
    'z_focal_plane_mm': scan_info['z_focal_plane'],
    'scan_params_by_m': scan_info['scan_params_by_m'],
    'm_angles': scan_info['m_angles'],
    'total_patterns': len(scan_info['patterns']),
    'total_frames': scan_info['total_frames'],
    'patterns': [
        {
            'name': p['name'],
            'm_pattern': p['m_pattern'],
            'num_steps': p['num_steps'],
            'z_range': p['z_range'],
            'folder': os.path.splitext(p['name'])[0],
        }
        for p in scan_info['patterns']
    ],
}

with open(master_log_path, 'w', encoding='utf-8') as f:
    json.dump(master_info, f, indent=2, ensure_ascii=False)

# ============================================================
# 打印汇总信息
# ============================================================
print(f"\n{'='*60}")
print(f"✅ 数据整理完成！")
print(f"   总扫描日志: {os.path.basename(master_log_path)}")
print(f"   创建文件夹: {len(scan_info['patterns'])} 个")
print(f"\n📊 各 M 值汇总:")
m_summary = {}
for p in scan_info['patterns']:
    m = p['m_pattern']
    if m not in m_summary:
        m_summary[m] = {'count': 0, 'frames': 0}
    m_summary[m]['count'] += 1
    m_summary[m]['frames'] += p['num_steps']

for m in sorted(m_summary.keys()):
    s = m_summary[m]
    print(f"   {m}: {s['count']} 个相位图, 共 {s['frames']} 帧")

print(f"\n📁 数据位置: {save_dir}")
print(f"{'='*60}")